# Employee Dataset Cleaning

Clean, standardize, validate, and save the messy employee dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Employee_Messy_20.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (20, 8)


## Standardize Text and Missing Values

In [15]:
str_cols = [
    col for col in df.columns
    if pd.api.types.is_string_dtype(df[col])
]
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
EmployeeID      0
EmployeeName    0
Gender          0
Department      0
Age             0
Salary          0
JoiningDate     0
City            0
dtype: int64


## Normalize Categorical Columns

In [ ]:
df["Gender"] = (
    df["Gender"]
    .str.upper()
    .map({"F": "Female", "FEMALE": "Female", "M": "Male", "MALE": "Male"})
)

print("Genders:")
print(df["Gender"].value_counts(dropna=False))

Genders:
Gender
Male      10
Female    10
Name: count, dtype: int64
Departments: ['Engineering', 'Finance', 'HR', 'Marketing', 'Sales']
Cities: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai', 'Pune']


In [ ]:
df["Department"] = df["Department"].str.title().replace("Hr", "HR")

print("Departments:", sorted(df["Department"].dropna().unique()))

In [ ]:
df["City"] = df["City"].str.title()

print("Cities:", sorted(df["City"].dropna().unique()))

## Clean Numeric Fields

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df.loc[~df["Age"].between(18, 65), "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median()).astype(int)

print(df["Age"].describe())

            Age        Salary
count  20.00000          20.0
mean   41.70000       59500.0
std     7.80081  15965.423165
min    22.00000       34000.0
25%    39.00000       45500.0
50%    44.00000       64500.0
75%    45.75000       71250.0
max    52.00000       81000.0


In [ ]:
salary_text = df["Salary"].astype("string").str.replace(",", "", regex=False)
salary_values = salary_text.str.extract(r"(\d+(?:\.\d+)?)$")[0]
df["Salary"] = pd.to_numeric(salary_values, errors="coerce")
df.loc[df["Salary"] <= 0, "Salary"] = np.nan
df["Salary"] = df["Salary"].fillna(df["Salary"].median()).round(2)

print(df["Salary"].describe())

## Normalize Joining Dates

In [18]:
df["JoiningDate"] = pd.to_datetime(
    df["JoiningDate"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

print("Invalid or missing dates:", df["JoiningDate"].isna().sum())
df["JoiningDate"] = df["JoiningDate"].dt.strftime("%Y-%m-%d")

Invalid or missing dates: 0


## Remove Duplicate Employees and Validate

In [19]:
duplicate_count = df["EmployeeID"].duplicated().sum()
df = df.drop_duplicates(subset="EmployeeID", keep="first").reset_index(drop=True)

assert df["EmployeeID"].is_unique
assert df["Gender"].dropna().isin(["Female", "Male"]).all()
assert df["Age"].between(18, 65).all()
assert df["Salary"].gt(0).all()
assert df["JoiningDate"].notna().all()

print("Duplicates removed:", duplicate_count)
print("Missing values:")
print(df.isna().sum())

Duplicates removed: 0
Missing values:
EmployeeID      0
EmployeeName    0
Gender          0
Department      0
Age             0
Salary          0
JoiningDate     0
City            0
dtype: int64


## Save the Cleaned Dataset

In [ ]:
output_path = "Employee_Cleaned_20.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Employee\Employee_Cleaned_20.csv
Saved shape: (20, 8)
